# **Ascend**

![Ascend logo](https://raw.githubusercontent.com/EduardoT22/Ascend/main/ascend-logo-transparent.png)

### RISE. TRACK. BECOME MORE.

**My app helps new Ascend users create a personalized fitness profile by validating their information, checking whether their goal weight aligns with their fitness goal, and estimating their daily calorie needs based on their personal metrics and activity level.**

Ascend started in MIST 2090 as a fitness and wellness app — workout discovery, daily progress tracking, and motivation in one place. In MIST 4600, this notebook builds the working part: new-user profile setup only. Workout logging, meal tracking, and wearables are out of scope here.

| # | Sub-question | Tools |
|---|---|---|
| 1 | **Goals:** how does the distribution of fitness goals compare among Ascend users? | pandas |
| 2 | **Activity:** how does activity level affect the average recommended daily calorie intake? | pandas |
| 3 | **Change:** among lose/gain users, what is the typical desired % weight change for each activity level? | pandas |
| 4 | *(Optional)* **Age:** how do fitness goals and average daily calorie needs differ across age groups? | pandas |

**Requirements checklist:** a user and a decision (above) · 3 sub-questions, all my own ideas · a function for each sub-question (`question_1`, `question_2`, `question_3`) · a class the app uses (`UserProfile`) · an outside tool (Gemini) · runs from top to bottom · a recommendation at the end.


---
## Before you run it

1. Put this notebook and `sample_profiles.csv` in the same folder in your Google Drive.
2. Add your Gemini key to Colab Secrets (key icon on the left) with the name `GOOGLE_API_KEY`, and turn on **Notebook access**. Get the key at [aistudio.google.com](https://aistudio.google.com) with a personal Gmail account.
3. Runtime → **Run all**. The Gemini cell calls Gemini once, so it takes a few seconds.


## Setup: find the data in Drive

Connects to Google Drive and finds the folder that holds `sample_profiles.csv`. The next cell uses that folder, `FOLDER`, to read the file. The sample profiles are 300 made-up profiles — no real users.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os

# Look for sample_profiles.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "sample_profiles.csv")

if len(found) == 0:
    print("❌ Can't find sample_profiles.csv in your Drive. Put it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    print("✅ Found the data in:", FOLDER)


## Cell 1 · Setup: libraries, data, and Gemini

Imports the libraries, reads the 300 sample profiles into `raw`, and connects to Gemini. The key is read from Colab Secrets, so it never appears in the code (this repository is public).


In [ ]:
import pandas as pd
import time
from google import genai
from google.colab import userdata

raw = pd.read_csv(FOLDER + "sample_profiles.csv")
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"

print(raw.shape)
print(raw.head())


## Cell 2 · The calorie math: Mifflin-St Jeor

This is the same formula calculator.net uses by default. BMR is calories at rest:
male: `10*kg + 6.25*cm - 5*age + 5`, female: same, but `-161` instead of `+5`.
Maintenance is BMR x activity: BMR 1.0, Sedentary 1.2, Light 1.375, Moderate 1.55, Active 1.725, Very Active 1.9, Extra Active 1.95 — calculator.net's 7 choices.
Goal-adjusted daily calories is maintenance, then lose -500, maintain 0, gain +500.
That ±500 is a simple documented rule, about 1 lb/week — my choice, not a medical recommendation.


In [ ]:
# Activity multipliers for the maintenance estimate.
ACTIVITY_MULTIPLIERS = {"bmr": 1.0, "sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9, "extra_active": 1.95}
GOAL_ADJUSTMENTS = {"lose weight": -500, "maintain weight": 0, "gain weight": 500}

def calculate_bmr(sex, age, height_inches, weight_lbs):
    # Takes sex, age, height (inches), weight (lb). Returns BMR in cal/day.
    weight_kg = weight_lbs / 2.20462
    height_cm = height_inches * 2.54
    if sex == "male":
        return 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    return 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

def estimate_maintenance_calories(sex, age, height_inches, weight_lbs, activity_level):
    # BMR x activity multiplier. Returns maintenance cal/day, rounded.
    bmr = calculate_bmr(sex, age, height_inches, weight_lbs)
    return round(bmr * ACTIVITY_MULTIPLIERS[activity_level])

def estimate_daily_calories(sex, age, height_inches, weight_lbs, activity_level, fitness_goal):
    # Goal-adjusted daily calories: maintenance + goal adjustment.
    maintenance = estimate_maintenance_calories(sex, age, height_inches, weight_lbs, activity_level)
    return maintenance + GOAL_ADJUSTMENTS[fitness_goal]

def desired_weight_change_pct(weight_lbs, goal_weight_lbs):
    # (goal - current) / current * 100. What the user WANTS to change, not what they lost/gained.
    return (goal_weight_lbs - weight_lbs) / weight_lbs * 100


## Cell 3 · Validation and the goal-weight check

`validate_profile` checks one profile dict and returns a list of error strings — empty means valid.
`check_goal_weight` checks that the goal weight agrees with the fitness goal: lose means goal < current, maintain means goal = current, gain means goal > current. This checks agreement, not medical appropriateness.


In [ ]:
FITNESS_GOALS = ("lose weight", "maintain weight", "gain weight")
ACTIVITY_LEVELS = ("bmr", "sedentary", "light", "moderate", "active", "very_active", "extra_active")
SEXES = ("male", "female")

def validate_profile(profile):
    errors = []
    name = profile.get("name")
    if not isinstance(name, str) or not name.strip():
        errors.append("name: enter a name for the profile.")
    age = profile.get("age")
    if isinstance(age, bool) or not isinstance(age, int) or age <= 0:
        errors.append("age: enter a positive whole number of years.")
    for field in ("height_inches", "weight_lbs", "goal_weight_lbs"):
        value = profile.get(field)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
            errors.append(f"{field}: enter a positive number.")
    if profile.get("fitness_goal") not in FITNESS_GOALS:
        errors.append("fitness_goal: choose one of lose weight / maintain weight / gain weight.")
    if profile.get("activity_level") not in ACTIVITY_LEVELS:
        errors.append("activity_level: choose one of bmr / sedentary / light / moderate / active / very_active / extra_active.")
    if profile.get("sex") not in SEXES:
        errors.append("sex: choose male or female.")
    return errors

def check_goal_weight(weight_lbs, goal_weight_lbs, fitness_goal):
    if fitness_goal == "lose weight":
        passes = goal_weight_lbs < weight_lbs
        rule = "goal weight must be below current weight to lose weight"
    elif fitness_goal == "maintain weight":
        passes = goal_weight_lbs == weight_lbs
        rule = "goal weight must equal current weight to maintain weight"
    elif fitness_goal == "gain weight":
        passes = goal_weight_lbs > weight_lbs
        rule = "goal weight must be above current weight to gain weight"
    else:
        return False, "fitness_goal is not one of lose / maintain / gain weight."
    if passes:
        return True, f"Goal weight {goal_weight_lbs} lb fits '{fitness_goal}' at {weight_lbs} lb."
    return False, f"Mismatch: {rule}, but you entered goal {goal_weight_lbs} lb at current {weight_lbs} lb. Please correct the goal or the goal weight."


## Cell 4 · A class for one profile

`UserProfile` holds the 8 proposal fields: name, age, height_inches, weight_lbs, fitness_goal, goal_weight_lbs, activity_level, sex.
`.validate()` and `.goal_check()` use the functions above. `.maintenance_calories`, `.estimated_daily_calories`, and `.desired_weight_change_pct` are the calculated results. `.to_dict()` adds the two calculated columns, ready for a pandas row.


In [ ]:
class UserProfile:
    def __init__(self, name, age, height_inches, weight_lbs,
                 fitness_goal, goal_weight_lbs, activity_level, sex):
        self.name = name
        self.age = age
        self.height_inches = height_inches
        self.weight_lbs = weight_lbs
        self.fitness_goal = fitness_goal
        self.goal_weight_lbs = goal_weight_lbs
        self.activity_level = activity_level
        self.sex = sex

    def as_dict(self):
        return {
            "name": self.name, "age": self.age,
            "height_inches": self.height_inches, "weight_lbs": self.weight_lbs,
            "fitness_goal": self.fitness_goal, "goal_weight_lbs": self.goal_weight_lbs,
            "activity_level": self.activity_level, "sex": self.sex,
        }

    def validate(self):
        return validate_profile(self.as_dict())

    def goal_check(self):
        return check_goal_weight(self.weight_lbs, self.goal_weight_lbs, self.fitness_goal)

    @property
    def maintenance_calories(self):
        return estimate_maintenance_calories(
            self.sex, self.age, self.height_inches, self.weight_lbs, self.activity_level)

    @property
    def estimated_daily_calories(self):
        return estimate_daily_calories(
            self.sex, self.age, self.height_inches, self.weight_lbs,
            self.activity_level, self.fitness_goal)

    @property
    def desired_weight_change_pct(self):
        return desired_weight_change_pct(self.weight_lbs, self.goal_weight_lbs)

    def to_dict(self):
        row = self.as_dict()
        row["estimated_daily_calories"] = self.estimated_daily_calories
        row["desired_weight_change_pct"] = round(self.desired_weight_change_pct, 2)
        return row


Try the class with one profile — the demo from the proposal: 20-year-old male, 72 in, 185 lb, lose to 176 lb, moderate activity.


In [ ]:
demo = UserProfile(
    name="Sample User", age=20, height_inches=72, weight_lbs=185,
    fitness_goal="lose weight", goal_weight_lbs=176,
    activity_level="moderate", sex="male",
)
print("Errors:", demo.validate())
print("Goal check:", demo.goal_check())
print("Maintenance calories:", demo.maintenance_calories)
print("Estimated daily calories (goal-adjusted, lose -500):", demo.estimated_daily_calories)
print("Desired change %:", round(demo.desired_weight_change_pct, 2))


## Cell 5 · Build the analysis table

`load_profiles` reads each sample row, builds a `UserProfile`, skips rows that fail validation or the goal check, and returns a DataFrame with the 8 inputs plus the 2 calculated columns. That is the table Q1–Q3 use, so the app and the analysis use the same calorie definition.


In [ ]:
def load_profiles(raw):
    rows = []
    for _, r in raw.iterrows():
        profile = UserProfile(
            name=r["name"], age=int(r["age"]), height_inches=float(r["height_inches"]),
            weight_lbs=float(r["weight_lbs"]), fitness_goal=r["fitness_goal"],
            goal_weight_lbs=float(r["goal_weight_lbs"]),
            activity_level=r["activity_level"], sex=r["sex"],
        )
        errors = profile.validate()
        passes, message = profile.goal_check()
        if errors:
            print(f"Skipping {r['name']}: validation errors {errors}")
            continue
        if not passes:
            print(f"Skipping {r['name']}: {message}")
            continue
        rows.append(profile.to_dict())
    return pd.DataFrame(rows)

df = load_profiles(raw)
print(f"Profiles analysed: {len(df)} (sample data only — no real users)")
print(df)


## Cell 6 · Sub-question 1: Goals

How does the distribution of fitness goals compare among Ascend users? Count + % per goal. `question_1` returns counts and percents.


In [ ]:
def question_1(df):
    counts = df["fitness_goal"].value_counts()
    pct = df["fitness_goal"].value_counts(normalize=True) * 100
    return pd.DataFrame({"count": counts, "percent": pct.round(1)})

print(question_1(df))


In [ ]:
# Brand chart for Q1 — Ascend logo colors, in order:
# pink #FF4D8D, magenta #D500E6, purple #7C2FF7, blue #2E7DFF.
import matplotlib.pyplot as plt

BRAND_COLORS = ["#FF4D8D", "#D500E6", "#B000F0", "#7C2FF7", "#5A3DFF", "#2E7DFF", "#00B4FF"]
q1 = question_1(df)
q1["count"].plot(kind="bar", color=BRAND_COLORS, figsize=(6, 4))
plt.title("Q1 — Fitness goal distribution")
plt.ylabel("Users")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Cell 7 · Sub-question 2: Activity

How does activity level affect the average recommended daily calorie intake? Group by activity level and compare the average estimated daily calories. Note: these are goal-adjusted averages, so a group average mixes activity AND the mix of lose / maintain / gain users in that group. Describe it as a difference in the app's goal-adjusted estimates, not as activity alone causing it.


In [ ]:
def question_2(df):
    return (
        df.groupby("activity_level")["estimated_daily_calories"]
        .agg(["count", "mean"])
        .round(1)
        .reindex(["bmr", "sedentary", "light", "moderate", "active", "very_active", "extra_active"])
    )

print(question_2(df))


In [ ]:
# Brand chart for Q2 — same Ascend gradient colors.
import matplotlib.pyplot as plt
q2 = question_2(df)
q2["mean"].plot(kind="bar", color=BRAND_COLORS, figsize=(6, 4))
plt.title("Q2 — Average goal-adjusted daily calories by activity level")
plt.ylabel("Calories / day")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Cell 8 · Sub-question 3: Change

Among users who want to lose or gain weight, what is the typical desired % weight change for each activity level? Keep lose and gain separate, group by activity level. "Typical" here is the median.


In [ ]:
def question_3(df):
    subset = df[df["fitness_goal"].isin(["lose weight", "gain weight"])]
    return (
        subset.groupby(["fitness_goal", "activity_level"])["desired_weight_change_pct"]
        .agg(["count", "median"])
        .round(2)
    )

print(question_3(df))


## Cell 9 · Sub-question 4 (optional): Age

How do fitness goals and average daily calorie needs differ across age groups? Age ranges are NOT in the proposal, so these are a draft choice: 18–24, 25–34, 35–49.


In [ ]:
def question_4(df):
    df = df.copy()
    df["age_group"] = pd.cut(df["age"], bins=[18, 25, 35, 50],
                             labels=["18-24", "25-34", "35-49"], right=False)
    goals = df.groupby(["age_group", "fitness_goal"], observed=True).size().unstack(fill_value=0)
    calories = df.groupby("age_group", observed=True)["estimated_daily_calories"].mean().round(1)
    return goals, calories

goals, calories = question_4(df)
print(goals)
print(calories)


## Cell 10 · Outside tool: Gemini recommendation

`get_recommendation` sends one new user's profile summary to Gemini and asks for one short, encouraging tip in one sentence. This is the outside tool at work — an LLM API called from the notebook. It helps answer the user's decision: what should this new user do with their calorie estimate?


In [ ]:
def get_recommendation(profile):
    time.sleep(4)
    prompt = (
        f"Write one short, encouraging tip in one sentence for a new Ascend user: "
        f"{profile.age} years old, {profile.sex}, {profile.weight_lbs} lb, "
        f"goal {profile.fitness_goal} to {profile.goal_weight_lbs} lb, "
        f"activity {profile.activity_level}, "
        f"goal-adjusted daily calories {profile.estimated_daily_calories}. "
        f"Do not give medical advice."
    )
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text.strip()

print(get_recommendation(demo))


---
## The recommendation

**For a new user:** Ascend validates the profile, checks the goal weight, and gives a goal-adjusted daily calorie target — not just maintenance. For the demo user (20, 72 in, 185 lb, lose to 176, moderate): BMR 1,887.1, maintenance 2,925, goal-adjusted 2,425 cal/day, desired change -4.86%.

**Sub-question 1:** lose 126 (42.0%), gain 105 (35.0%), maintain 69 (23.0%) of the 300 sample profiles.
**Sub-question 2:** average goal-adjusted daily calories — BMR 1,614.4 (n=42), Sedentary 1,897.8 (n=43), Light 2,024.4 (n=43), Moderate 2,456.2 (n=44), Active 2,674.3 (n=43), Very Active 2,879.4 (n=43), Extra Active 2,955.2 (n=42).
**Sub-question 3:** median desired % change — lose -10.00% (BMR, n=17), -7.03% (Sedentary, n=18), -8.27% (Light, n=18), -9.09% (Moderate, n=19), -8.95% (Active, n=18), -8.20% (Very Active, n=18), -8.53% (Extra Active, n=18); gain +6.81% (BMR, n=15), +5.19% (Sedentary, n=15), +7.14% (Light, n=15), +5.77% (Moderate, n=15), +6.86% (Active, n=15), +6.98% (Very Active, n=15), +5.99% (Extra Active, n=15).

**For Ascend:** start every new user with this profile setup, show them the goal-adjusted target and whether their goal weight fits their goal, and use the sample answers above to explain what typical users look like. Workout logging, meal tracking, and wearables are the next steps, out of scope for this project.
